# 📦 1️⃣ Importaciones y carga de claves

In [23]:
import os
import json
import numpy as np
from pathlib import Path
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()

client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

# -----------------------------
# RUTAS CORRECTAS PARA JUPYTER
# -----------------------------

# Ruta del proyecto (sube 1 nivel desde /notebooks)
BASE_DIR = Path().resolve().parents[0]      # <-- esta es la clave

# Si tu notebook está en: 03_IA_Segunda_Guerra_Mundial/notebooks/
# Entonces BASE_DIR apunta a: 03_IA_Segunda_Guerra_Mundial/

PROCESSED_DIR = BASE_DIR / "data" / "processed"
CHUNKS_PATH = PROCESSED_DIR / "ww2_chunks_plus_qa.jsonl"
EMB_PATH = PROCESSED_DIR / "ww2_embeddings_plus_qa.npy"


# 📄 2️⃣ Cargar chunks y embeddings

In [24]:
def cargar_datos():
    chunks = []
    with open(CHUNKS_PATH, "r", encoding="utf-8") as f:
        for line in f:
            chunks.append(json.loads(line))

    emb_matrix = np.load(EMB_PATH)

    print(f"Chunks cargados: {len(chunks)}")
    print(f"Embeddings shape: {emb_matrix.shape}")

    return chunks, emb_matrix


chunks, emb_matrix = cargar_datos()


Chunks cargados: 4937
Embeddings shape: (4937, 3072)


# 🧠 3️⃣ Función para crear embeddings de la pregunta

In [25]:
def embed_text(text):
    resp = client.embeddings.create(
        model="text-embedding-3-large",
        input=text,
    )
    return np.array(resp.data[0].embedding, dtype="float32")


# 📏 4️⃣ Similitud coseno + recuperación de top-k fragmentos

In [26]:
def cosine_sim(query_vec, docs_matrix):
    num = docs_matrix @ query_vec
    den = (np.linalg.norm(docs_matrix, axis=1) * np.linalg.norm(query_vec) + 1e-10)
    return num / den


def recuperar_chunks(query, k=5):
    q_emb = embed_text(query)
    sims = cosine_sim(q_emb, emb_matrix)

    idx = np.argsort(sims)[::-1][:k]

    resultados = []
    for i in idx:
        c = chunks[int(i)]
        resultados.append({
            "score": float(sims[i]),
            "text": c["text"],
            "source": c["source"],
            "page": c["page"]
        })

    return resultados


# 📝 5️⃣ Prompt conciso + llamada al modelo

In [ ]:
def expandir_siglas(texto: str) -> str:
    """
    Sustituye siglas por su explicación la primera vez que aparecen en el texto.
    Solo expande la primera aparición para evitar repeticiones pesadas.
    """

    sustituciones = {
        "RAF": "RAF (Real Fuerza Aérea británica)",
        "Luftwaffe": "Luftwaffe (fuerza aérea alemana)",
        "Kriegsmarine": "Kriegsmarine (marina de guerra alemana)",
        "Wehrmacht": "Wehrmacht (fuerzas armadas alemanas)",
        "USAAF": "USAAF (Fuerza Aérea del Ejército de EE.UU.)",
    }

    for sigla, expansion in sustituciones.items():
        # Solo sustituye la PRIMERA aparición de cada sigla
        texto = texto.replace(sigla, expansion, 1)

    return texto


def generar_respuesta(query, contexto):
    # Construir fragmentos de contexto
    partes = []
    for i, c in enumerate(contexto):
        partes.append(
            f"[Fragmento {i+1} | Fuente: {c['source']} pág. {c['page']}]\n{c['text']}"
        )
    context_text = "\n\n".join(partes)

    mensajes = [
        {
            "role": "system",
            "content": (
                "Eres un historiador experto en la Segunda Guerra Mundial. "
                "Respondes siempre en español, de manera breve, clara y natural, máximo 6 líneas. "
                "Tu única fuente es el contexto proporcionado. No inventes datos. "
                "Evita expresiones demasiado técnicas o rígidas."
            ),
        },
        {
            "role": "user",
            "content": (
                f"{context_text}\n\n"
                "Usa SOLO la información anterior. Si algo no aparece, responde: "
                "\"No existe información suficiente en los documentos para responder.\"\n\n"
                "Instrucciones:\n"
                "- Máximo 6 líneas.\n"
                "- Sin subtítulos.\n"
                "- Respuesta clara y concisa.\n"
                "- Evita frases demasiado técnicas o robóticas.\n"
                "- Ajusta el tono para que sea natural, no académico.\n"
                "- Añade citas al final: (Fuente: <PDF>, pág. X).\n\n"
                f"Pregunta: {query}"
            ),
        },
    ]

    resp = client.chat.completions.create(
        model="gpt-4.1-mini",
        messages=mensajes,
        max_completion_tokens=300,
    )

    respuesta = resp.choices[0].message.content
    # Postproceso: expandir siglas (RAF, Luftwaffe, etc.) la primera vez que aparezcan
    respuesta = expandir_siglas(respuesta)
    return respuesta


# ❓ 6️⃣ Función final para preguntarle:

In [28]:
def responder(query, k=5):
    contexto = recuperar_chunks(query, k)
    respuesta = generar_respuesta(query, contexto)
    return respuesta


# PREGUNTAR

In [29]:
responder("¿Por qué fue importante la Operación Barbarroja?")


'La Operación Barbarroja fue importante porque representó el plan de la Alemania nazi de conquistar la Unión Soviética occidental para repoblarla con alemanes y obtener recursos vitales como petróleo del Cáucaso y productos agrícolas de Ucrania. Además, implicaba el exterminio, esclavización y deportación masiva de los pueblos eslavos para ganar espacio vital ("lebensraum"). Aunque inicialmente logró victorias y capturas masivas, fracasó en derrotar rápidamente a la Unión Soviética y se estancó en la batalla de Moscú. (Fuente: Frente_oriental, pág. 6; Operación_Barbarroja, págs. 0, 1, 28)'

In [30]:
responder("¿Cuáles fueron las principales causas de la Segunda Guerra Mundial?")

'Las principales causas de la Segunda Guerra Mundial fueron las tensiones no resueltas tras la Primera Guerra Mundial, la inestabilidad generada por el Tratado de Versalles, la crisis económica de entreguerras, y la política expansionista y racista de Hitler. Además, la incapacidad de las potencias para frenar a tiempo la agresión alemana y la política de apaciguamiento de países como Reino Unido facilitaron el crecimiento de la influencia de Alemania en Europa Central. Hitler, con sus acciones agresivas, desencadenó el conflicto. (Fuente: qa_manual pág.1; Frente_oriental_SGM.pdf pág.2; Segunda_Guerra_Mundial.pdf pág.1)'

In [31]:
responder("¿Cuándo empezó la segunda guerra mundial?")

'La Segunda Guerra Mundial comenzó el 1 de septiembre de 1939 con la invasión alemana de Polonia. La madrugada de ese día, bombarderos alemanes atacaron infraestructuras claves y el acorazado Schleswig-Holstein comenzó a abrir fuego. Dos días después, Reino Unido y Francia le declararon la guerra a Alemania. La Unión Soviética invadió Polonia el 17 de septiembre de 1939. (Fuente: Segunda_Guerra_Mundial.pdf, pág. 0, 1, 4; Invasión_de_Polonia.pdf, pág. 0).'

In [32]:
responder("¿Cómo afectó la relación entre Hitler y sus generales al desarrollo de la guerra?")

'La relación entre Hitler y sus generales se deterioró por la desconfianza mutua y la tendencia de Hitler a asumir el mando directo, ignorando consejos de oficiales experimentados como Rundstedt y Manstein. Hitler redujo la autonomía del Estado Mayor y tomó decisiones tácticas rígidas, prohibiendo retiradas necesarias. Esta interferencia constante y los cambios de objetivos trastornaron a sus generales, contribuyendo a errores tácticos y estratégicos graves. (Fuente: qa_manual, pág. 1; Operación_Barbarroja.pdf, págs. 10, 59).'

In [33]:
responder("¿Por qué la batalla de Inglaterra fue importante para frenar la expansión alemana?")

'La batalla de Inglaterra fue importante porque Alemania nazi intentó destruir a la Real Fuerza Aérea británica (RAF) para obtener superioridad aérea y poder invadir Gran Bretaña. Sin embargo, la derrota alemana impidió esa invasión y mantuvo a Gran Bretaña como resistencia ante Alemania. Además, fue la primera derrota alemana en la Segunda Guerra Mundial y evitó que la Kriegsmarine, la marina alemana, enfrentara riesgos mayores. Esto puso freno a la expansión alemana en el frente occidental. (Fuente: Batalla_de_Inglaterra.pdf, págs. 0-1)'